# FDM expansion for Diffusive Equations

In [50]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
import sympy as sp
import numpy as np
from IPython.display import display, Math, Latex

us = sp.symbols('u_{x}^{t}')
omega = sp.symbols(r'\omega')
omg1=1-omega
cs = sp.symbols('c_s')

In [51]:
def display_broken_equation(equation, terms_per_line=4):
    lhs = sp.latex(equation.lhs)

    # Separate the additive terms on the right-hand side
    # rhs_terms = sp.Add.make_args(sp.expand(equation.rhs))
    rhs_terms = sp.Add.make_args(equation.rhs)

    lines = []

    for i in range(0, len(rhs_terms), terms_per_line):
        group = rhs_terms[i:i + terms_per_line]
        expression = sp.Add(*group)

        latex_group = sp.latex(expression)

        if i == 0:
            lines.append(rf"{lhs} &= {latex_group}")
        else:
            lines.append(rf"&\quad {latex_group}")

    latex_output = (
        r"\begin{aligned}"
        + r" \\ ".join(lines)
        + r"\end{aligned}"
    )

    display(Math(latex_output))

 ## One-Dimensional Cases: $D1Qq$ Lattices

In [31]:
import sympy as sp
ul = {}
past_steps = [1, 2, 3, 4]
space_offsets = [1, 2, 3, 4]
# time_map: key_prefix -> (latex_expr): "+1" -> u^{t+1},  "0" -> u^{t},  "-1".."-4" -> u^{t-k}
time_map = {"+1": "t+1","0":  "t",**{f"-{k}": f"t-{k}" for k in past_steps},}

# Non-shifted symbols
for tk, te in time_map.items():
    ul[f"ut{tk}"] = sp.Symbol(fr'u^{{{te}}}_{{x}}')

# Shifted symbols (skip the future step "+1")
for tk, te in time_map.items():
    if tk == "+1":
        continue
    for ssign in ("+", "-"):
        for d in space_offsets:
            ul[f"ut{tk}d{ssign}{d}"] = sp.Symbol(fr'u^{{{te}}}_{{x{ssign}{d}}}')

In [33]:
# ul

In [4]:
def make_shift_symbols(prefix=r'T_{\Delta_{x}}', order=9, var='c', start=0):
    names = [f'{prefix}^{{{var}_{{{i}}}}}' for i in range(start, start + order)]
    return sp.symbols(names)
Txc = make_shift_symbols(order=9, var='c')   # list of SymPy symbols
# Txc

In [5]:
from itertools import combinations, combinations_with_replacement
# ----Multi-index shift symbols with permutation deduplication------
def make_perm_shift_symbols(prefix=r'T_{\Delta_{x}}',
                            order=9,
                            var='c',
                            start=0,
                            length=2,
                            include_repeats=False,
                            canonical=True):
    indices = range(start, start + order)
    if include_repeats:
        combos = combinations_with_replacement(indices, length)
    else:
        combos = combinations(indices, length)
    out = {}
    for combo in combos:
        key = tuple(sorted(combo)) if canonical else tuple(combo)
        sup = ''.join(f'{var}_{{{i}}}' for i in key)
        out[key] = sp.Symbol(fr'{prefix}^{{{sup}}}')
    return out

In [6]:
p2T = make_perm_shift_symbols(order=5, length=2, include_repeats=False)
# print(p2T[(0, 1)])   # T_{\Delta_{x}}^{c_{0}c_{1}}
print(len(p2T))      # 36

10


In [7]:
p3T = make_perm_shift_symbols(order=5, length=3, include_repeats=False)
# print(p2T[(0,1,2)])   # T_{\Delta_{x}}^{c_{0}c_{1}}
print(len(p3T))      # 36

10


In [8]:
def wheigt_symbols(prefix=r'w', order=9, start=0):
    names = [f'{prefix}_{{{i}}}' for i in range(start, start + order)]
    return sp.symbols(names)
w = wheigt_symbols(order=9)   # list of SymPy symbols
# w[0]

### D1Q3

In [9]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np
q=3
cx=np.array([0,1,-1])
wis=np.array([w[0],w[1],w[2]])
feq = us*wis

In [10]:
OrderT = Matrix([0, 0, 0])
# First order (single index)
for i0 in range(q):
    OrderT[0] = OrderT[0] + omega * feq[i0] * Txc[i0] + (1 - omega) * Txc[i0] * us

# Second order (unordered pairs i0 < i1)
for i0 in range(q):
    for i1 in range(i0 + 1, q):
        OrderT[1] = OrderT[1] + omega*(feq[1]+feq[i0]) * Txc[i0]*Txc[i1] + (1-omega)*Txc[i0]*Txc[i1]*us

# Third order (unordered triples i0 < i1 < i2)
for i0 in range(q):
    for i1 in range(i0 + 1, q):
        for i2 in range(i1 + 1, q):
            OrderT[2] = OrderT[2] + omega*(feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2] + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*us

In [11]:
# display(OrderT[0])
mt0=sp.collect(OrderT[0]
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut0d-1'])
           .subs(Txc[2]*us,ul['ut0d+1'])
           .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{us,ul['ut0d-1'],ul['ut0d+1']}
          )
display(mt0)

u^{t}_{x+1}*(1 - 5*\omega/6) + u^{t}_{x-1}*(1 - 5*\omega/6) + u_{x}^{t}*(1 - \omega/3)

In [12]:
# display(OrderT[1])
mt1=sp.collect(sp.expand(-OrderT[1]*omg1)
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut-1d-1'])
           .subs(Txc[2]*us,ul['ut-1d+1'])
           .subs(us,ul['ut-1'])
           .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-1'],ul['ut-1d-1'],ul['ut-1d+1']}
          )
display(mt1)

u^{t-1}_{x+1}*(-\omega**2/6 + 7*\omega/6 - 1) + u^{t-1}_{x-1}*(-\omega**2/6 + 7*\omega/6 - 1) + u^{t-1}_{x}*(-2*\omega**2/3 + 5*\omega/3 - 1)

In [13]:
mt2=sp.simplify(sp.expand(OrderT[2]*(omg1**2))
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs({w[1]:Rational(1,6),w[2]:Rational(1,6),w[0]:Rational(2,3)})
           .subs(us,ul['ut-2'])
          )
display(mt2)

u^{t-2}_{x}*(\omega**2 - 2*\omega + 1)

In [14]:
sp.Eq(ul['ut+1'],mt0+mt1+mt2)

Eq(u^{t+1}_{x}, u^{t-1}_{x+1}*(-\omega**2/6 + 7*\omega/6 - 1) + u^{t-1}_{x-1}*(-\omega**2/6 + 7*\omega/6 - 1) + u^{t-1}_{x}*(-2*\omega**2/3 + 5*\omega/3 - 1) + u^{t-2}_{x}*(\omega**2 - 2*\omega + 1) + u^{t}_{x+1}*(1 - 5*\omega/6) + u^{t}_{x-1}*(1 - 5*\omega/6) + u_{x}^{t}*(1 - \omega/3))

### D1Q5ZOT

In [54]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np
q=5
cx=np.array([0,1,-1,3,-3])
wis=np.array([w[0],w[1],w[2],w[3],w[4]])
w0 = sp.Rational(16, 45) + sp.Rational(4, 45)*sp.sqrt(10)
w1 = sp.Rational(3, 10) - sp.Rational(3, 80)*sp.sqrt(10)
w2 = sp.Rational(1, 45) - sp.Rational(1, 144)*sp.sqrt(10)
feq = us*wis

In [55]:
OrderT = Matrix([0, 0, 0, 0, 0])
# First order (single index)
for i0 in range(q):
    OrderT[0] = OrderT[0] + omega * feq[i0] * Txc[i0] + (1 - omega) * Txc[i0] * us

# Second order (unordered pairs i0 < i1)
for i0 in range(q):
    for i1 in range(i0 + 1, q):
        OrderT[1] = OrderT[1] + omega*(feq[1]+feq[i0]) * Txc[i0]*Txc[i1] + (1-omega)*Txc[i0]*Txc[i1]*us

# Third order (unordered triples i0 < i1 < i2)
for i0 in range(q):
    for i1 in range(i0 + 1, q):
        for i2 in range(i1 + 1, q):
            OrderT[2] = OrderT[2] + omega*(feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2] + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*us

# Fourth order (unordered triples i0 < i1 < i2 < i3)
for i0 in range(q):
    for i1 in range(i0 + 1, q):
        for i2 in range(i1 + 1, q):
            for i3 in range(i2 + 1, q):
                OrderT[3] = OrderT[3] + (omega*(feq[i3]+feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3] 
                                         + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3]*us)

# Fiveth order (unordered triples i0 < i1 < i2 < i3 < i4)
for i0 in range(q):
    for i1 in range(i0 + 1, q):
        for i2 in range(i1 + 1, q):
            for i3 in range(i2 + 1, q):
                for i4 in range(i3 + 1, q):
                    OrderT[4] = OrderT[4] + (omega*(feq[i4]+feq[i3]+feq[i2]+feq[i1]+feq[i0]) *Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3]*Txc[i4] 
                                         + (1-omega)*Txc[i0]*Txc[i1]*Txc[i2]*Txc[i3]*Txc[i4]*us)

In [56]:
# display(OrderT[0])
mt0=sp.collect(OrderT[0]
           .subs({Txc[1]*Txc[2]:1,Txc[0]:1})
           .subs(Txc[1]*us,ul['ut0d-1'])
           .subs(Txc[2]*us,ul['ut0d+1'])
           .subs(Txc[3]*us,ul['ut0d-3'])
           .subs(Txc[4]*us,ul['ut0d+3'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{us,ul['ut0d-1'],ul['ut0d+1'],ul['ut0d-3'],ul['ut0d+3']}
          )
display(mt0)

u^{t}_{x+1}*(\omega*w_{2} - \omega + 1) + u^{t}_{x+3}*(\omega*w_{4} - \omega + 1) + u^{t}_{x-1}*(\omega*w_{1} - \omega + 1) + u^{t}_{x-3}*(\omega*w_{3} - \omega + 1) + u_{x}^{t}*(\omega*w_{0} - \omega + 1)

In [57]:
display(OrderT[1])
mt1=sp.collect(sp.expand(OrderT[1]*(-omg1))
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1,
                  Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-1d-1'])
           .subs(Txc[2]*us,ul['ut-1d+1'])
           .subs(Txc[3]*us,ul['ut-1d-3'])
           .subs(Txc[4]*us,ul['ut-1d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-1d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-1d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-1d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-1d-2'])
           .subs(us,ul['ut-1'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-1'],ul['ut-1d-1'],ul['ut-1d+1'],ul['ut-1d-2'],ul['ut-1d+2'],ul['ut-1d-3'],ul['ut-1d+3'],ul['ut-1d-4'],ul['ut-1d+4']}
          )
display(mt1)

T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{3}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{3}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{4}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{4}}*u_{x}^{t}*(1 - \omega) + 2*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_{1} + T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}*(1 - \omega) + 2*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{3}}*\omega*u_{x}^{t}*w_{1} + T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{3}}*u_{x}^{t}*(1 - \omega) + 2*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{4}}*\omega*u_{x}^{t}

u^{t-1}_{x+1}*(\omega**2*w_{0} + \omega**2*w_{1} - \omega**2 - \omega*w_{0} - \omega*w_{1} + 2*\omega - 1) + u^{t-1}_{x+2}*(2*\omega**2*w_{1} - \omega**2 - 2*\omega*w_{1} + 2*\omega - 1) + u^{t-1}_{x+3}*(\omega**2*w_{0} + \omega**2*w_{1} - \omega**2 - \omega*w_{0} - \omega*w_{1} + 2*\omega - 1) + u^{t-1}_{x+4}*(\omega**2*w_{1} + \omega**2*w_{2} - \omega**2 - \omega*w_{1} - \omega*w_{2} + 2*\omega - 1) + u^{t-1}_{x-1}*(\omega**2*w_{0} + \omega**2*w_{1} - \omega**2 - \omega*w_{0} - \omega*w_{1} + 2*\omega - 1) + u^{t-1}_{x-2}*(\omega**2*w_{1} + \omega**2*w_{2} - \omega**2 - \omega*w_{1} - \omega*w_{2} + 2*\omega - 1) + u^{t-1}_{x-3}*(\omega**2*w_{0} + \omega**2*w_{1} - \omega**2 - \omega*w_{0} - \omega*w_{1} + 2*\omega - 1) + u^{t-1}_{x-4}*(2*\omega**2*w_{1} - \omega**2 - 2*\omega*w_{1} + 2*\omega - 1) + u^{t-1}_{x}*(3*\omega**2*w_{1} + \omega**2*w_{3} - 2*\omega**2 - 3*\omega*w_{1} - \omega*w_{3} + 4*\omega - 2)

In [58]:
display(OrderT[2])
# mt1_0=(sp.expand(OrderT[2]*(-omg1)**2)
#            .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1})
#            .subs({Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
#            )
# display(mt1_0)
mt2=sp.collect(sp.expand(OrderT[2]*(-omg1)**2)
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-2d-1'])
           .subs(Txc[2]*us,ul['ut-2d+1'])
           .subs(Txc[3]*us,ul['ut-2d-3'])
           .subs(Txc[4]*us,ul['ut-2d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-2d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-2d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-2d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-2d-2'])
           .subs(us,ul['ut-2'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-2'],ul['ut-2d-1'],ul['ut-2d+1'],ul['ut-2d-2'],ul['ut-2d+2'],ul['ut-2d-3'],ul['ut-2d+3'],ul['ut-2d-4'],ul['ut-2d+4']}
          )
display(mt2)

T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1} + u_{x}^{t}*w_{2}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{3}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1} + u_{x}^{t}*w_{3}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{3}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{4}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1} + u_{x}^{t}*w_{4}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{4}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{3}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{2} + u_{x}^{t}*w_{3}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{3}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{4}}*\omega*(u_{x}^{t}*w_{0} + u

u^{t-2}_{x+1}*(\omega**3*w_{2} + \omega**3*w_{3} + \omega**3*w_{4} - \omega**3 - 2*\omega**2*w_{2} - 2*\omega**2*w_{3} - 2*\omega**2*w_{4} + 3*\omega**2 + \omega*w_{2} + \omega*w_{3} + \omega*w_{4} - 3*\omega + 1) + u^{t-2}_{x+2}*(\omega**3*w_{0} + \omega**3*w_{1} + \omega**3*w_{4} - \omega**3 - 2*\omega**2*w_{0} - 2*\omega**2*w_{1} - 2*\omega**2*w_{4} + 3*\omega**2 + \omega*w_{0} + \omega*w_{1} + \omega*w_{4} - 3*\omega + 1) + u^{t-2}_{x+3}*(\omega**3*w_{1} + \omega**3*w_{2} + \omega**3*w_{4} - \omega**3 - 2*\omega**2*w_{1} - 2*\omega**2*w_{2} - 2*\omega**2*w_{4} + 3*\omega**2 + \omega*w_{1} + \omega*w_{2} + \omega*w_{4} - 3*\omega + 1) + u^{t-2}_{x+4}*(\omega**3*w_{0} + \omega**3*w_{2} + \omega**3*w_{4} - \omega**3 - 2*\omega**2*w_{0} - 2*\omega**2*w_{2} - 2*\omega**2*w_{4} + 3*\omega**2 + \omega*w_{0} + \omega*w_{2} + \omega*w_{4} - 3*\omega + 1) + u^{t-2}_{x-1}*(\omega**3*w_{1} + \omega**3*w_{3} + \omega**3*w_{4} - \omega**3 - 2*\omega**2*w_{1} - 2*\omega**2*w_{3} - 2*\omega**2*w_{

In [59]:
display(OrderT[3])
mt3=sp.collect(sp.expand(OrderT[3]*(-omg1)**3)
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-3d-1'])
           .subs(Txc[2]*us,ul['ut-3d+1'])
           .subs(Txc[3]*us,ul['ut-3d-3'])
           .subs(Txc[4]*us,ul['ut-3d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-3d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-3d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-3d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-3d-2'])
           .subs(us,ul['ut-3'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-3'],ul['ut-3d-1'],ul['ut-3d+1'],ul['ut-3d-2'],ul['ut-3d+2'],ul['ut-3d-3'],ul['ut-3d+3'],ul['ut-3d-4'],ul['ut-3d+4']}
          )
display(mt3)

T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{3}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1} + u_{x}^{t}*w_{2} + u_{x}^{t}*w_{3}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{3}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{4}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1} + u_{x}^{t}*w_{2} + u_{x}^{t}*w_{4}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{4}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{3}}*T_{\Delta_{x}}^{c_{4}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1} + u_{x}^{t}*w_{3} + u_{x}^{t}*w_{4}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{3}}*T_{\Delta_{x}}^{c_{4}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{3}}*T_{\Delta_{x}}^{c_{4}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_

u^{t-3}_{x+1}*(\omega**4*w_{0} + \omega**4*w_{2} + \omega**4*w_{3} + \omega**4*w_{4} - \omega**4 - 3*\omega**3*w_{0} - 3*\omega**3*w_{2} - 3*\omega**3*w_{3} - 3*\omega**3*w_{4} + 4*\omega**3 + 3*\omega**2*w_{0} + 3*\omega**2*w_{2} + 3*\omega**2*w_{3} + 3*\omega**2*w_{4} - 6*\omega**2 - \omega*w_{0} - \omega*w_{2} - \omega*w_{3} - \omega*w_{4} + 4*\omega - 1) + u^{t-3}_{x+3}*(\omega**4*w_{0} + \omega**4*w_{1} + \omega**4*w_{2} + \omega**4*w_{4} - \omega**4 - 3*\omega**3*w_{0} - 3*\omega**3*w_{1} - 3*\omega**3*w_{2} - 3*\omega**3*w_{4} + 4*\omega**3 + 3*\omega**2*w_{0} + 3*\omega**2*w_{1} + 3*\omega**2*w_{2} + 3*\omega**2*w_{4} - 6*\omega**2 - \omega*w_{0} - \omega*w_{1} - \omega*w_{2} - \omega*w_{4} + 4*\omega - 1) + u^{t-3}_{x-1}*(\omega**4*w_{0} + \omega**4*w_{1} + \omega**4*w_{3} + \omega**4*w_{4} - \omega**4 - 3*\omega**3*w_{0} - 3*\omega**3*w_{1} - 3*\omega**3*w_{3} - 3*\omega**3*w_{4} + 4*\omega**3 + 3*\omega**2*w_{0} + 3*\omega**2*w_{1} + 3*\omega**2*w_{3} + 3*\omega**2*w_{4} - 6

In [60]:
display(OrderT[4])
mt4=sp.collect(sp.expand(OrderT[4]*(-omg1)**4)
           .subs({Txc[1]*Txc[2]:1, Txc[3]*Txc[4]:1, Txc[0]:1})
           .subs({Txc[1]*Txc[3]:p2T[(1, 3)], Txc[2]*Txc[4]:p2T[(2, 4)], Txc[1]*Txc[4]:p2T[(1, 4)], Txc[2]*Txc[3]:p2T[(2, 3)]})
           .subs(Txc[1]*us,ul['ut-4d-1'])
           .subs(Txc[2]*us,ul['ut-4d+1'])
           .subs(Txc[3]*us,ul['ut-4d-3'])
           .subs(Txc[4]*us,ul['ut-4d+3'])        
           .subs(p2T[(1, 3)]*us,ul['ut-4d-4'])
           .subs(p2T[(2, 4)]*us,ul['ut-4d+4'])
           .subs(p2T[(1, 4)]*us,ul['ut-4d+2'])
           .subs(p2T[(2, 3)]*us,ul['ut-4d-2'])
           .subs(us,ul['ut-4'])
           # .subs({w[2]:Rational(1,6),w[1]:Rational(1,6),w[0]:Rational(2,3)})
           ,{ul['ut-4'],ul['ut-4d-1'],ul['ut-4d+1'],ul['ut-4d-2'],ul['ut-4d+2'],ul['ut-4d-3'],ul['ut-4d+3'],ul['ut-4d-4'],ul['ut-4d+4']}
          )
display(mt4)

T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{3}}*T_{\Delta_{x}}^{c_{4}}*\omega*(u_{x}^{t}*w_{0} + u_{x}^{t}*w_{1} + u_{x}^{t}*w_{2} + u_{x}^{t}*w_{3} + u_{x}^{t}*w_{4}) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*T_{\Delta_{x}}^{c_{3}}*T_{\Delta_{x}}^{c_{4}}*u_{x}^{t}*(1 - \omega)

u^{t-4}_{x}*(\omega**5*w_{0} + \omega**5*w_{1} + \omega**5*w_{2} + \omega**5*w_{3} + \omega**5*w_{4} - \omega**5 - 4*\omega**4*w_{0} - 4*\omega**4*w_{1} - 4*\omega**4*w_{2} - 4*\omega**4*w_{3} - 4*\omega**4*w_{4} + 5*\omega**4 + 6*\omega**3*w_{0} + 6*\omega**3*w_{1} + 6*\omega**3*w_{2} + 6*\omega**3*w_{3} + 6*\omega**3*w_{4} - 10*\omega**3 - 4*\omega**2*w_{0} - 4*\omega**2*w_{1} - 4*\omega**2*w_{2} - 4*\omega**2*w_{3} - 4*\omega**2*w_{4} + 10*\omega**2 + \omega*w_{0} + \omega*w_{1} + \omega*w_{2} + \omega*w_{3} + \omega*w_{4} - 5*\omega + 1)

In [63]:
def brokeq(equation, terms_per_line=4):
    lhs = sp.latex(equation.lhs)
    rhs_terms = list(sp.Add.make_args(equation.rhs))

    lines = []
    for i in range(0, len(rhs_terms), terms_per_line):
        group = rhs_terms[i:i + terms_per_line]

        # Build the latex for this group manually so we control signs
        terms_latex = []
        for j, term in enumerate(group):
            coeff, rest = term.as_coeff_Mul()
            sign = "-" if coeff < 0 else "+"
            # strip sign from latex
            term_latex = sp.latex(abs(coeff) * rest) if coeff != 1 and coeff != -1 \
                         else sp.latex(rest)
            terms_latex.append((sign, term_latex))

        # First term of the whole RHS has no leading + / -
        first_sign, first_tex = terms_latex[0]
        if i == 0:
            head = first_tex if first_sign == "+" else f"- {first_tex}"
            line = rf"{lhs} &= {head}"
        else:
            head = first_tex if first_sign == "+" else f"- {first_tex}"
            prefix = "+ " if first_sign == "+" else ""
            # Actually for continuation we want the sign explicit:
            line = rf"&\quad {first_sign} {first_tex}"

        # Remaining terms in the group
        for sign, tex in terms_latex[1:]:
            line += rf" {sign} {tex}"

        lines.append(line)

    return r"\begin{aligned}" + r" \\ ".join(lines) + r"\end{aligned}"

In [62]:
display_broken_equation(sp.Eq(ul['ut+1'],mt0+mt1+mt2+mt3+mt4), terms_per_line=1)

<IPython.core.display.Math object>

In [66]:
# display(Math(brokeq(sp.Eq(ul['ut+1'],mt0+mt1+mt2+mt3+mt4), terms_per_line=1)))